In [ ]:
import scanpy as sc
import pandas as pd

group_files = {
    "post_nres_ASDC":        "adata_post_nres_ASDC.h5ad",
    "post_nres_MPPpreB":     "adata_post_nres_MPPpreB.h5ad",
    "post_nres_HSCLSC":      "adata_post_nres_HSCLSC.h5ad",
    "post_nres_MPPCLP1":     "adata_post_nres_MPPCLP1.h5ad",
    "post_nres_EMP":         "adata_post_nres_EMP.h5ad",
    "post_nres_Macrophage1": "adata_post_nres_Macrophage1.h5ad",
    "post_nres_LE":          "adata_post_nres_LE.h5ad",
    "post_nres_GMP1":        "adata_post_nres_GMP1.h5ad",
    "post_nres_Erythroid":   "adata_post_nres_Erythroid.h5ad",
}

rows = []

for ct, f in group_files.items():
    adata = sc.read_h5ad(f)
    tmp = (
        adata.obs
        .groupby("patient")
        .size()
        .reset_index(name="n_cells")
    )
    tmp["cell_type"] = ct
    rows.append(tmp)

df_counts = pd.concat(rows, ignore_index=True)


In [ ]:
df_counts["total_cells"] = (
    df_counts
    .groupby("cell_type")["n_cells"]
    .transform("sum")
)

df_counts["patient_ratio"] = (
    df_counts["n_cells"] / df_counts["total_cells"]
)

df_counts


In [ ]:
target_cts = [
    "post_nres_ASDC",
    "post_nres_MPPpreB",
    "post_nres_MPPCLP1"
]

df_counts["group_class"] = df_counts["cell_type"].apply(
    lambda x: "target" if x in target_cts else "other"
)
patient_matrix = (
    df_counts
    .pivot_table(
        index="cell_type",
        columns="patient",
        values="patient_ratio",
        fill_value=0
    )
)

patient_matrix


In [ ]:
import numpy as np
from scipy.spatial.distance import jensenshannon

def js_dist(a, b):
    return jensenshannon(a, b)

dist_results = []

for ct in target_cts:
    v_ct = patient_matrix.loc[ct]
    v_other = patient_matrix.drop(ct).mean(axis=0)
    dist = js_dist(v_ct, v_other)
    dist_results.append((ct, dist))

pd.DataFrame(dist_results, columns=["cell_type", "JS_distance"])


In [ ]:
from scipy.stats import chi2_contingency

stats = []

for ct in target_cts:
    sub = df_counts.copy()
    sub["is_ct"] = sub["cell_type"] == ct

    table = (
        sub
        .pivot_table(
            index="patient",
            columns="is_ct",
            values="n_cells",
            aggfunc="sum",
            fill_value=0
        )
    )

    chi2, p, _, _ = chi2_contingency(table)
    stats.append((ct, p))

pd.DataFrame(stats, columns=["cell_type", "chi2_pvalue"])


In [ ]:
import pandas as pd

groups_to_check = [
    "post_nres_ASDC",
    "post_nres_MPPpreB",
    "post_nres_MPPCLP1"
]

patient_dist = []

for g in groups_to_check:
    adata = adatas[g]
    
    df = (
        adata.obs
        .groupby("patient")
        .size()
        .reset_index(name="n_cells")
    )
    df["group"] = g
    patient_dist.append(df)

patient_dist_df = pd.concat(patient_dist, ignore_index=True)
patient_dist_df
